# Changing the Schrödinger state

We have already seen that a qubit state lies on the circumference of the unit circle, and that projectors and observables act on that state. Now we introduce the formalism for how states themselves are changed.

Operations on a qubit's state are represented by **unitary matrices**. A matrix $U$ is unitary if $UU^\dagger = I$ — its conjugate transpose is also its inverse.

**Why must state transformations be unitary?**

1. **Norm preservation.** A qubit state must remain normalised: its squared amplitudes must always sum to 1, since they represent probabilities. Unitary matrices preserve the length of every vector they act on, keeping the state on the unit circle after any transformation. A non-unitary gate would stretch or shrink the state vector, producing amplitudes whose squares no longer sum to 1.

2. **Reversibility.** Quantum evolution outside of measurement is always reversible. Because $U^\dagger = U^{-1}$, applying $U$ can always be undone by applying $U^\dagger$. This means quantum information is never destroyed by a gate. Non-invertible matrices would permanently discard information and violate the time-reversibility of quantum mechanics.

3. **Inner product preservation.** Unitary matrices preserve inner products as well as lengths: $\bra{\phi}U^\dagger U\ket{\psi} = \braket{\phi|\psi}$. This keeps the relative angles between states intact, which is what allows quantum interference — the wave-like cancellations and reinforcements that make quantum computing powerful — to work correctly.

**The Hadamard gate**

The Hadamard gate is an important single-qubit gate:

$$H = \frac{1}{\sqrt{2}} \begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$$

It converts $\ket{0}$ into $\ket{+}$ and $\ket{1}$ into $\ket{-}$:

$$H\ket{0} = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix} = \ket{+} \qquad\qquad H\ket{1} = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix} = \ket{-}$$

Intuitively, the Hadamard rotates the unit circle so that the $\ket{0}/\ket{1}$ axis and the $\ket{+}/\ket{-}$ axis swap. Because of this symmetry, applying it twice returns every state to where it started: $H^2 = I$, so the Hadamard gate is its own inverse.


# Changing the Heisenberg state

There are two equivalent ways to describe a transformation:

- Apply the unitary $U$ to the state and keep the observables fixed — the **Schrödinger picture**.
- Keep the state fixed and transform each observable as $O \mapsto U^\dagger\,O\,U$ — the **Heisenberg picture**.

**Deriving the Heisenberg picture**

Say the system is in state $\ket{\psi}$ and we apply a unitary $U$ — a rotation, time evolution, a change of basis, whatever it may be. The new state is:

$$\ket{\psi'} = U\ket{\psi}$$

We want the expectation value of observable $O$ to be the same whether we transform the state or transform the operator:

$$\bra{\psi'} O \ket{\psi'} = \bra{\psi} O' \ket{\psi}$$

Expanding the left side:

$$\bra{\psi'} O \ket{\psi'} = \bra{U\psi} O \ket{U\psi} = \bra{\psi} U^\dagger O U \ket{\psi}$$

So the transformed observable must be:

$$O' = U^\dagger O U$$

As a concrete example, consider starting in state $\ket{0}$. $\sigma_z$ measures in the computational basis; $\ket{0}$ is an eigenstate of $\sigma_z$ with eigenvalue $+1$, so $\bra{0}\sigma_z\ket{0} = 1$. After applying the Hadamard, the state becomes $\ket{+}$, equally weighted between $\ket{0}$ and $\ket{1}$, so measuring with $\sigma_z$ gives an expectation value of 0 — we no longer know whether the outcome will be $+1$ or $-1$. In the Heisenberg picture we get the same result by keeping the state as $\ket{0}$ and instead transforming the observable: $H^\dagger\,\sigma_z\,H = \sigma_x$, and $\bra{0}\sigma_x\ket{0} = 0$.

**The Hadamard gate swaps a qubit's X and Z observables**

The Hadamard matrix is real and symmetric, so $H^\dagger = H^T = H$. Applying the formula $O' = H^\dagger O H$:

$$H^\dagger\,\sigma_z\,H = \frac{1}{\sqrt{2}}\begin{bmatrix}1&1\\1&-1\end{bmatrix}\begin{bmatrix}1&0\\0&-1\end{bmatrix}\frac{1}{\sqrt{2}}\begin{bmatrix}1&1\\1&-1\end{bmatrix} = \begin{bmatrix}0&1\\1&0\end{bmatrix} = \sigma_x$$

The Z observable becomes the X observable. Applying the same formula to $\sigma_x$ gives back $\sigma_z$. The Hadamard gate swaps the two observables.

Applying Hadamard twice means sandwiching twice. The second application undoes the first:

$$H^\dagger(H^\dagger\,\sigma_z\,H)H = H^\dagger\,\sigma_x\,H = \sigma_z$$

The observables return to their starting values, consistent with $H^2 = I$ in the Schrödinger picture.

In [1]:
from matrix_definitions import hadamard, xobservable, zobservable
from sympy import simplify
from IPython.display import display

# H† σ_z H: the Z observable becomes the X observable
display(simplify(hadamard.H * zobservable * hadamard))

# H† σ_x H: the X observable becomes the Z observable
display(simplify(hadamard.H * xobservable * hadamard))

Matrix([
[0, 1],
[1, 0]])

Matrix([
[1,  0],
[0, -1]])

In [13]:
# Applying the Hadamard gate twice restores the original observables
after_one_h = simplify(hadamard.H * zobservable * hadamard)
after_two_h = simplify(hadamard.H * after_one_h * hadamard)
display(after_two_h)

Matrix([
[1,  0],
[0, -1]])

# Pre-computed gate results in `gates_single.py`

The matrix calculations above establish the effect of each gate on each observable. Once that relationship is known and tested, we no longer need to think in terms of matrices at all. `gates_single.py` encodes these results directly as transformations of a `Qubit`'s observables, letting us reason about gates at a higher level of abstraction: the Hadamard gate swaps X and Z (and negates Y).

For the Hadamard gate the three results,
* $H\,\sigma_z\,H^T = \sigma_x$
* $H\,\sigma_x\,H^T = \sigma_z$
* $H\,\sigma_y\,H^T = -\sigma_y$

are captured in a single function:

```python
def hadamard(qubit: Qubit):
    return qubit.with_observables(qubit.z, -qubit.y, qubit.x)
```

The function takes a `Qubit` and returns a new `Qubit` with updated observables.

**Tests prove the equivalence**

`tests/test_gate.py` verifies that each pre-computed gate matches the result of evaluating the sandwich formula via `Qubit.evolve()`. For the Hadamard gate, the test constructs the unitary as the Pauli expression $\frac{1}{\sqrt{2}}(\sigma_x + \sigma_z)$, evolves a qubit through it symbolically, and asserts that the result equals the pre-computed function's output. Separate tests confirm the matrix sandwich $H^T\,\sigma\,H$ gives the expected Pauli matrix for each of the X, Y, and Z observables.

With the equivalence established by tests, the rest of the code can stay at the level of *which gates are applied to which qubits*. Reasoning about a quantum circuit becomes a matter of tracking how observables transform step by step — no matrices required.

In [14]:
from qubit import Qubit
from gates_single import hadamard

qubit = Qubit.qubit_time_0('a')
display(qubit)

# The Hadamard gate swaps X and Z and negates Y
display(hadamard(qubit))

# Applying the Hadamard gate twice returns the qubit to its original state
display(hadamard(hadamard(qubit)))

Qubit(a, a1, a2, a3)

Qubit(a, a3, -a2, a1)

Qubit(a, a1, a2, a3)